In [ ]:
%pip install lakebench[tpcds_datagen]

In [ ]:
# enabling v-order
spark.conf.set("spark.sql.parquet.vorder.enabled", "true")

# Enable optimized writes to reduce small files
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")

# Set the target bin size for optimized writes to 1GB
spark.conf.set("spark.databricks.delta.optimizeWrite.binSize", "1g")

# Set ZSTD compression codec explicitly for your write operation
spark.conf.set("spark.sql.parquet.compression.codec", "zstd")

In [ ]:
import ipywidgets as widgets

dropdown = widgets.Dropdown(
    options=[10, 100,1000],
    value=1, # Default value
    description='SF',
    disabled=False,
)

display(dropdown)

In [ ]:
from lakebench.datagen import TPCDSDataGenerator
import duckdb


temp_dir = f'/tmp/duckdb_tmp_sf{dropdown.value}'                 # set as needed depending on the compute size used to generate the data

base_path=f'Files/tpch_sf{dropdown.value}'

_original_duckdb_connect = duckdb.connect

def _connect_with_temp_dir(*args, **kwargs):
    con = _original_duckdb_connect(*args, **kwargs)
    con.execute(f"SET temp_directory='{temp_dir}'")              # set as needed depending on the compute size used to generate the data
    con.execute(f"SET memory_limit='400GB'")                     # set as needed depending on the compute size used to generate the data
    return con

try:
    duckdb.connect = _connect_with_temp_dir
    datagen = TPCDSDataGenerator(
        scale_factor=dropdown.value,
        target_folder_uri=f'/lakehouse/default/{base_path}'
    )
    datagen.run()
finally:
    duckdb.connect = _original_duckdb_connect

In [ ]:
tables = ['catalog_page', 'catalog_sales', 'customer_address', 'customer_demographics', 'date_dim', 'item', 'promotion', 'ship_mode', 'store', 'store_sales']

int_columns = [
    'd_date_sk', 'd_month_seq', 'd_week_seq', 'd_quarter_seq', 'd_year', 
    'd_dow', 'd_moy', 'd_dom', 'd_qoy', 'd_fy_year', 'd_fy_quarter_seq', 
    'd_fy_week_seq', 'd_first_dom', 'd_last_dom', 'd_same_day_ly', 'd_same_day_lq','cp_catalog_page_sk', 'cp_start_date_sk','cp_end_date_sk','cp_catalog_number','cp_catalog_page_number','ca_address_sk','cd_demo_sk','cd_purchase_estimate','cd_dep_count','cd_dep_employed_count','cd_dep_college_count','i_item_sk','i_brand_id','i_class_id','i_category_id','i_manufact_id','i_manager_id','p_promo_sk','p_start_date_sk','p_end_date_sk','p_item_sk','p_response_target','sm_ship_mode_sk','s_store_sk','s_closed_date_sk','s_number_employees','s_floor_space','s_market_id','s_division_id','s_company_id','cs_call_center_sk','cs_catalog_page_sk','cs_ship_mode_sk','cs_bill_customer_sk','cs_bill_cdemo_sk','cs_bill_hdemo_sk','cs_bill_addr_sk','cs_ship_customer_sk','cs_ship_cdemo_sk','cs_ship_hdemo_sk','cs_ship_addr_sk','cs_promo_sk','cs_item_sk','cs_quantity',
    'cs_sold_date_sk','cs_sold_time_sk','cs_ship_date_sk','cs_warehouse_sk','ss_sold_date_sk','ss_sold_time_sk','ss_item_sk','ss_customer_sk','ss_cdemo_sk','ss_hdemo_sk','ss_addr_sk','ss_store_sk','ss_promo_sk','ss_quantity'
]

# Change bigint to int and save as delta tables
for table in tables:
    df = spark.read.parquet(f"{volume_path}/{table}")
    for col in int_columns:
        if col in df.columns and dict(df.dtypes)[col] == 'bigint':
            df = df.withColumn(col, df[col].cast('int'))
    df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog_name}.{schema_name}.{table}")

In [ ]:
# Amend date_dim to bring time forward and delete unused years
# Add d_date_sk_1 as the first column and overwrite the table
df = spark.table("date_dim")
df = df.withColumn("d_date_sk_1", df["d_date_sk"] - 8527)
cols = ["d_date_sk_1"] + [col for col in df.columns if col != "d_date_sk_1"]
df = df.select(cols)
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("date_dim")

# Delete rows where d_date is < "2021-01-01" or > "2026-12-31"
spark.sql("""
DELETE FROM date_dim
WHERE d_date < '2021-01-01' OR d_date > '2026-12-31'
""")

In [ ]:
# Delete Nulls to ensure Referential Integrity on Facts
# Delete Nulls from catalog_sales
spark.sql(f"""
DELETE FROM catalog_sales
WHERE {' OR '.join([f'cs_{col} IS NULL' for col in ['sold_date_sk','sold_time_sk','ship_date_sk','bill_customer_sk','bill_cdemo_sk','bill_hdemo_sk','bill_addr_sk','ship_customer_sk','ship_cdemo_sk','ship_hdemo_sk','ship_addr_sk','call_center_sk','catalog_page_sk','ship_mode_sk','warehouse_sk','item_sk','promo_sk','order_number','quantity','wholesale_cost','list_price','sales_price','ext_discount_amt','ext_sales_price','ext_wholesale_cost','ext_list_price','ext_tax','coupon_amt','ext_ship_cost','net_paid','net_paid_inc_tax','net_paid_inc_ship','net_paid_inc_ship_tax','net_profit']])}
""")

# Delete Nulls from store_sales
spark.sql(f"""
DELETE FROM store_sales
WHERE {' OR '.join([f'ss_{col} IS NULL' for col in ['sold_date_sk','sold_time_sk','item_sk','customer_sk','cdemo_sk','hdemo_sk','addr_sk','store_sk','promo_sk','ticket_number','quantity','wholesale_cost','list_price','sales_price','ext_discount_amt','ext_sales_price','ext_wholesale_cost','ext_list_price','ext_tax','coupon_amt','net_paid','net_paid_inc_tax','net_profit']])}
""")

In [ ]:
# Add a cache_buster column for load testing
for table in ['catalog_sales', 'store_sales']:
    spark.sql(f"""
        ALTER TABLE {table}
        ADD COLUMN cache_buster INT
    """)
    spark.sql(f"""
        UPDATE {table}
        SET cache_buster = 1
    """)

In [ ]:
# Partition and Z-order store_sales
df=spark.read.table("store_sales")

df.write.format("delta").mode("overwrite").partitionBy("ss_sold_date_sk").saveAsTable("store_sales")

spark.sql("OPTIMIZE store_sales ZORDER BY (ss_addr_sk)")

In [ ]:
# Partition and Z-order catalog_sales
df=spark.read.table("catalog_sales")

df.write.format("delta").mode("overwrite").partitionBy("cs_sold_date_sk").saveAsTable("catalog_sales")

spark.sql("OPTIMIZE catalog_sales ZORDER BY (cs_bill_addr_sk)")

In [3]:
# Run VACUUM and ANALYZE on Fact Tables
fact_tables = ['catalog_sales', 'store_sales']

for fact_table in fact_tables:
    spark.sql(f"VACUUM {fact_table}")
    spark.sql(f"ANALYZE TABLE {fact_table} COMPUTE STATISTICS FOR ALL COLUMNS")

StatementMeta(, a118c8ed-ce9d-451d-9ec8-f6f19c4964c4, 5, Finished, Available, Finished, False)

In [4]:
# Run OPTIMIZE, VACUUM and ANALYZE on Dim Tables
dim_tables = ['catalog_page', 'customer_address', 'customer_demographics', 'date_dim', 'item', 'promotion', 'ship_mode', 'store']

for dim_table in dim_tables:
    spark.sql(f"OPTIMIZE {dim_table}")
    spark.sql(f"VACUUM {dim_table}")
    spark.sql(f"ANALYZE TABLE {dim_table} COMPUTE STATISTICS FOR ALL COLUMNS")

StatementMeta(, a118c8ed-ce9d-451d-9ec8-f6f19c4964c4, 6, Finished, Available, Finished, False)